In [1]:
import pandas as pd

path = "../data/raw/Mobile_Parts_Wholesale_Dataset.xlsx"
po = pd.read_excel(path, sheet_name="Purchase_Orders")
suppliers = pd.read_excel(path, sheet_name="Suppliers")

po['order_date'] = pd.to_datetime(po['order_date'])
po['promised_delivery_date'] = pd.to_datetime(po['promised_delivery_date'])
po['actual_delivery_date'] = pd.to_datetime(po['actual_delivery_date'])

delivered = po[po['status'] != 'Pending'].copy()
delivered['on_time'] = delivered['actual_delivery_date'] <= delivered['promised_delivery_date']

scorecard = delivered.groupby('supplier_id').agg(
    total_orders=('po_id', 'count'),
    on_time_pct=('on_time', 'mean'),
    avg_unit_cost=('unit_cost_inr', 'mean')
).reset_index()

scorecard = scorecard.merge(suppliers[['supplier_id','supplier_name','country']], on='supplier_id')
scorecard['on_time_pct'] = (scorecard['on_time_pct'] * 100).round(1)

scorecard.to_csv("../reports/supplier_scorecard.csv", index=False)
print("Saved supplier_scorecard.csv")
scorecard.sort_values('on_time_pct', ascending=False)

Saved supplier_scorecard.csv


,supplier_id,total_orders,on_time_pct,avg_unit_cost,supplier_name,country
8,S009,19,78.9,518.955263,Sathe Ltd Trading Co.,China
1,S002,25,76.0,707.209600,"Dass, Magar and Dugar Trading Co.",India
4,S005,16,75.0,406.018125,"Gala, Agate and Ravi Trading Co.",India
3,S004,16,75.0,460.857500,Varughese-Kata Trading Co.,China
12,S013,16,75.0,523.118750,Sarma LLC Electronics,India
5,S006,23,73.9,412.950870,Sekhon-Radhakrishnan Electronics,India
6,S007,21,71.4,354.422857,Goda-Anand Trading Co.,China
11,S012,29,69.0,438.362414,"Das, Misra and Bava Electronics",China
2,S003,19,68.4,475.937895,Oommen Ltd Trading Co.,China
7,S008,21,66.7,1028.004286,Sachar-Sen Electronics,China
